In [2]:
import os

data_path = "data/clean_midi"
artists = os.listdir(data_path)
print("Number of artists:", len(artists))
print("First 10 artists:", artists[:10])

Number of artists: 2200
First 10 artists: ['.38 Special', '10,000 Maniacs', '101 Strings', '10cc', '1910 Fruitgum Company', '2 Brothers on the 4th Floor', '2 the Core', '2 Unlimited', '20 Fingers', '2Boys']


In [3]:
import pretty_midi
import os

# Pick a real artist and see what songs they have
artist = "10cc"
artist_path = os.path.join(data_path, artist)
songs = os.listdir(artist_path)
print("Songs by", artist, ":", songs)

Songs by 10cc : ['Dreadlock Holiday.1.mid', 'Dreadlock Holiday.2.mid', 'Dreadlock Holiday.3.mid', 'Dreadlock Holiday.4.mid', 'Dreadlock Holiday.mid', "I'm Not In Love.1.mid", "I'm Not In Love.2.mid", "I'm Not In Love.3.mid", "I'm Not In Love.mid", 'The Things We Do for Love.mid']


In [3]:
song_path = os.path.join(artist_path, "Dreadlock Holiday.mid")

midi_data = pretty_midi.PrettyMIDI(song_path)

print("Song duration (seconds):", midi_data.get_end_time())
print("Number of instruments:", len(midi_data.instruments))

for i, instrument in enumerate(midi_data.instruments):
    instrument_name = pretty_midi.program_to_instrument_name(instrument.program)
    print(f"Instrument {i}: {instrument_name} | Drum track: {instrument.is_drum} | Notes: {len(instrument.notes)}")

Song duration (seconds): 307.736506875
Number of instruments: 9
Instrument 0: Acoustic Grand Piano | Drum track: True | Notes: 3282
Instrument 1: Acoustic Grand Piano | Drum track: False | Notes: 473
Instrument 2: Acoustic Grand Piano | Drum track: False | Notes: 456
Instrument 3: Acoustic Grand Piano | Drum track: False | Notes: 2851
Instrument 4: Acoustic Grand Piano | Drum track: False | Notes: 3369
Instrument 5: Acoustic Grand Piano | Drum track: False | Notes: 1626
Instrument 6: Acoustic Grand Piano | Drum track: False | Notes: 2939
Instrument 7: Acoustic Grand Piano | Drum track: False | Notes: 244
Instrument 8: Acoustic Grand Piano | Drum track: False | Notes: 426


In [4]:
# Find the non-drum instrument with the most notes
melodic_instruments = [inst for inst in midi_data.instruments if not inst.is_drum]
main_instrument = max(melodic_instruments, key=lambda inst: len(inst.notes))

print("Chosen instrument index:", midi_data.instruments.index(main_instrument))
print("Number of notes:", len(main_instrument.notes))

# Look at the first 10 notes
for note in main_instrument.notes[:10]:
    print(f"Pitch: {note.pitch}, Start: {note.start:.2f}s, End: {note.end:.2f}s, Velocity: {note.velocity}")

Chosen instrument index: 4
Number of notes: 3369
Pitch: 62, Start: 4.52s, End: 6.17s, Velocity: 27
Pitch: 70, Start: 4.61s, End: 6.18s, Velocity: 60
Pitch: 74, Start: 4.65s, End: 6.18s, Velocity: 91
Pitch: 67, Start: 4.58s, End: 6.18s, Velocity: 64
Pitch: 62, Start: 6.31s, End: 6.62s, Velocity: 32
Pitch: 72, Start: 6.40s, End: 6.64s, Velocity: 73
Pitch: 65, Start: 6.35s, End: 6.65s, Velocity: 64
Pitch: 69, Start: 6.37s, End: 6.66s, Velocity: 60
Pitch: 58, Start: 6.94s, End: 9.24s, Velocity: 40
Pitch: 63, Start: 6.97s, End: 9.24s, Velocity: 73


In [5]:
# Extract melody: keep only the highest-pitch note at each unique start time
from collections import defaultdict

notes_by_start = defaultdict(list)
for note in main_instrument.notes:
    notes_by_start[round(note.start, 2)].append(note)

melody_notes = []
for start_time in sorted(notes_by_start.keys()):
    notes_at_time = notes_by_start[start_time]
    highest_note = max(notes_at_time, key=lambda n: n.pitch)
    melody_notes.append(highest_note)

print("Original note count:", len(main_instrument.notes))
print("Melody-only note count:", len(melody_notes))

# Preview first 10 melody notes
for note in melody_notes[:10]:
    print(f"Pitch: {note.pitch}, Start: {note.start:.2f}s, Duration: {note.end - note.start:.2f}s")

Original note count: 3369
Melody-only note count: 2292
Pitch: 62, Start: 4.52s, Duration: 1.65s
Pitch: 67, Start: 4.58s, Duration: 1.60s
Pitch: 70, Start: 4.61s, Duration: 1.57s
Pitch: 74, Start: 4.65s, Duration: 1.53s
Pitch: 62, Start: 6.31s, Duration: 0.31s
Pitch: 65, Start: 6.35s, Duration: 0.31s
Pitch: 69, Start: 6.37s, Duration: 0.29s
Pitch: 72, Start: 6.40s, Duration: 0.23s
Pitch: 58, Start: 6.94s, Duration: 2.29s
Pitch: 63, Start: 6.97s, Duration: 2.27s


In [6]:
# Group notes into chords using a time tolerance window
tolerance = 0.05  # 50ms — notes within this window count as the same chord

sorted_notes = sorted(main_instrument.notes, key=lambda n: n.start)

melody_notes = []
i = 0
while i < len(sorted_notes):
    current_group = [sorted_notes[i]]
    j = i + 1
    while j < len(sorted_notes) and sorted_notes[j].start - sorted_notes[i].start < tolerance:
        current_group.append(sorted_notes[j])
        j += 1
    highest_note = max(current_group, key=lambda n: n.pitch)
    melody_notes.append(highest_note)
    i = j

print("Original note count:", len(main_instrument.notes))
print("Melody-only note count:", len(melody_notes))

for note in melody_notes[:10]:
    print(f"Pitch: {note.pitch}, Start: {note.start:.2f}s, Duration: {note.end - note.start:.2f}s")

Original note count: 3369
Melody-only note count: 1058
Pitch: 62, Start: 4.52s, Duration: 1.65s
Pitch: 70, Start: 4.61s, Duration: 1.57s
Pitch: 74, Start: 4.65s, Duration: 1.53s
Pitch: 65, Start: 6.35s, Duration: 0.31s
Pitch: 72, Start: 6.40s, Duration: 0.23s
Pitch: 70, Start: 6.98s, Duration: 2.26s
Pitch: 74, Start: 9.31s, Duration: 1.69s
Pitch: 72, Start: 11.05s, Duration: 0.29s
Pitch: 63, Start: 11.63s, Duration: 2.33s
Pitch: 70, Start: 11.65s, Duration: 2.30s


In [7]:

tolerance = 0.15  # widened to 150ms

sorted_notes = sorted(main_instrument.notes, key=lambda n: n.start)

melody_notes = []
i = 0
while i < len(sorted_notes):
    current_group = [sorted_notes[i]]
    j = i + 1
    while j < len(sorted_notes) and sorted_notes[j].start - sorted_notes[i].start < tolerance:
        current_group.append(sorted_notes[j])
        j += 1
    highest_note = max(current_group, key=lambda n: n.pitch)
    melody_notes.append(highest_note)
    i = j

print("Original note count:", len(main_instrument.notes))
print("Melody-only note count:", len(melody_notes))

for note in melody_notes[:10]:
    print(f"Pitch: {note.pitch}, Start: {note.start:.2f}s, Duration: {note.end - note.start:.2f}s")



Original note count: 3369
Melody-only note count: 933
Pitch: 74, Start: 4.65s, Duration: 1.53s
Pitch: 72, Start: 6.40s, Duration: 0.23s
Pitch: 70, Start: 6.98s, Duration: 2.26s
Pitch: 74, Start: 9.31s, Duration: 1.69s
Pitch: 72, Start: 11.05s, Duration: 0.29s
Pitch: 70, Start: 11.65s, Duration: 2.30s
Pitch: 55, Start: 13.98s, Duration: 0.26s
Pitch: 70, Start: 14.31s, Duration: 0.12s
Pitch: 70, Start: 14.44s, Duration: 0.01s
Pitch: 70, Start: 14.86s, Duration: 0.06s


In [4]:
def extract_melody_from_midi(midi_path, tolerance=0.15):
    """
    Loads a MIDI file and extracts a melody-only note sequence
    (collapses chords into their highest-pitch note).
    Returns None if the file can't be processed (corrupted/unreadable).
    """
    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        return None  # some Lakh files are corrupted or malformed

    # pick non-drum instrument with the most notes
    melodic_instruments = [inst for inst in midi_data.instruments if not inst.is_drum]
    if not melodic_instruments:
        return None  # no melodic instrument found

    main_instrument = max(melodic_instruments, key=lambda inst: len(inst.notes))
    if len(main_instrument.notes) == 0:
        return None

    sorted_notes = sorted(main_instrument.notes, key=lambda n: n.start)

    melody_notes = []
    i = 0
    while i < len(sorted_notes):
        current_group = [sorted_notes[i]]
        j = i + 1
        while j < len(sorted_notes) and sorted_notes[j].start - sorted_notes[i].start < tolerance:
            current_group.append(sorted_notes[j])
            j += 1
        highest_note = max(current_group, key=lambda n: n.pitch)
        melody_notes.append(highest_note)
        i = j

    return melody_notes


# Test it on the same song to confirm it matches our earlier output
test_path = os.path.join(data_path, "10cc", "Dreadlock Holiday.mid")
result = extract_melody_from_midi(test_path)
print("Notes extracted:", len(result))
for note in result[:5]:
    print(f"Pitch: {note.pitch}, Start: {note.start:.2f}s")

Notes extracted: 933
Pitch: 74, Start: 4.65s
Pitch: 72, Start: 6.40s
Pitch: 70, Start: 6.98s
Pitch: 74, Start: 9.31s
Pitch: 72, Start: 11.05s


In [5]:
import glob

# Find all .mid files across all artists (this just lists paths, doesn't load them yet)
all_midi_files = glob.glob(os.path.join(data_path, "**", "*.mid"), recursive=True)
print("Total MIDI files found:", len(all_midi_files))

# Take a small sample to start
sample_files = all_midi_files[:50]
print("Sample size:", len(sample_files))
print("First few files:", sample_files[:3])

Total MIDI files found: 17167
Sample size: 50
First few files: ['data/clean_midi\\10,000 Maniacs\\A Campfire Song.mid', 'data/clean_midi\\101 Strings\\Theme From The Godfather.mid', 'data/clean_midi\\10cc\\Dreadlock Holiday.1.mid']


In [7]:
processed_melodies = []
failed_files = []

for file_path in sample_files:
    melody = extract_melody_from_midi(file_path)
    if melody is not None and len(melody) > 0:
        processed_melodies.append({"file": file_path, "melody": melody})
    else:
        failed_files.append(file_path)

print(f"Successfully processed: {len(processed_melodies)} / {len(sample_files)}")
print(f"Failed: {len(failed_files)}")
if failed_files:
    print("Example failed files:", failed_files[:5])

Successfully processed: 49 / 50
Failed: 1
Example failed files: ['data/clean_midi\\10cc\\Dreadlock Holiday.4.mid']


*49/50 succeeded — that's a healthy 98% success rate, and exactly the kind of real-world messiness we expected (Lakh is known to have some corrupted/unusual files). Our error handling did its job — the pipeline didn't crash, it just skipped the bad file and kept going. *"built the pipeline defensively since real datasets have corrupted files, and measured a 98% success rate on a sample."

Progress Log update
Built reusable extract_melody_from_midi() function — wraps chord-to-melody extraction with error handling for corrupted/malformed files.
Tested on a sample of 50 songs — 49/50 processed successfully (98% success rate), confirming the pipeline handles real-world messy data gracefully.

In [8]:
# Collect all pitches across all processed melodies
all_pitches = set()
for item in processed_melodies:
    for note in item["melody"]:
        all_pitches.add(note.pitch)

# Build vocabulary: pitch -> integer token
vocab = sorted(all_pitches)
pitch_to_token = {pitch: i for i, pitch in enumerate(vocab)}
token_to_pitch = {i: pitch for i, pitch in enumerate(vocab)}

print("Vocabulary size (unique pitches):", len(vocab))
print("Pitch range:", min(vocab), "to", max(vocab))
print("Sample mapping:", {k: pitch_to_token[k] for k in list(pitch_to_token)[:5]})

Vocabulary size (unique pitches): 65
Pitch range: 25 to 93
Sample mapping: {25: 0, 27: 1, 28: 2, 29: 3, 30: 4}


In [9]:
# Convert each song's melody (list of Note objects) into a list of integer tokens
tokenized_songs = []
for item in processed_melodies:
    tokens = [pitch_to_token[note.pitch] for note in item["melody"]]
    tokenized_songs.append(tokens)

print("Number of tokenized songs:", len(tokenized_songs))
print("First song length (notes):", len(tokenized_songs[0]))
print("First 20 tokens of first song:", tokenized_songs[0][:20])

Number of tokenized songs: 49
First song length (notes): 373
First 20 tokens of first song: [48, 40, 40, 41, 40, 38, 38, 40, 38, 36, 36, 40, 40, 40, 38, 38, 36, 40, 40, 43]


In [10]:
sequence_length = 50  # how many past notes the model sees before predicting the next

input_sequences = []
target_notes = []

for tokens in tokenized_songs:
    if len(tokens) <= sequence_length:
        continue  # skip songs too short to form even one training example
    for i in range(len(tokens) - sequence_length):
        input_sequences.append(tokens[i : i + sequence_length])
        target_notes.append(tokens[i + sequence_length])

print("Total training examples:", len(input_sequences))
print("Example input (first 10 of 50 tokens):", input_sequences[0][:10])
print("Example target (the note that follows):", target_notes[0])

Total training examples: 26377
Example input (first 10 of 50 tokens): [48, 40, 40, 41, 40, 38, 38, 40, 38, 36]
Example target (the note that follows): 40
